![](https://upload.wikimedia.org/wikipedia/commons/thumb/f/fd/RMS_Titanic_3.jpg/1200px-RMS_Titanic_3.jpg)

## Introduction

This project focuses on predicting the survival of passengers aboard the RMS Titanic, which sank in 1912. The dataset contains demographic and socio-economic information about the passengers, such as gender, age, ticket class, fare, and port of embarkation.

During the data preprocessing stage, missing values were handled using median, mean, and mode imputation techniques, while irrelevant or heavily missing columns (such as Cabin) were removed from the dataset. Feature engineering included encoding categorical variables (like Sex and Embarked) into numeric form using LabelEncoder and filtering out extreme outliers to ensure model stability.

For model training, a LightGBM Classifier was employed. LightGBM, being a gradient boosting framework, provides high efficiency and strong predictive performance. The dataset was split into training and testing subsets to evaluate generalization, and hyperparameter tuning was applied using GridSearchCV to optimize model performance.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

import warnings
from warnings import filterwarnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
filterwarnings("ignore")

## 1-Load Datasets

In [ ]:
train_df=pd.read_csv("/kaggle/input/titanic/train.csv")
test_df=pd.read_csv("/kaggle/input/titanic/test.csv")

## 2- Analyzing Datasets

In [ ]:
train_df.info()

In [ ]:
test_df.info()

We don't need "PassengerId","Ticket","Name" columns. Because they aren't importance for survived. But I saved my "PassengerId" column of test dataset because we would use that final csv file.

In [ ]:
train_df=train_df.drop(columns=["PassengerId","Ticket", "Name"])
test_df=test_df.drop(columns=["Name","Ticket"])

In [ ]:
train_df.head(5)

## 3-EDA

In [ ]:
sns.barplot(data=train_df,x="Sex",y="Survived")
plt.title("Survival Rate by Gender")
plt.ylabel("Survival Rate")
plt.xlabel("Gender")
plt.show()

In [ ]:
sns.catplot(data=train_df,x="Pclass",hue="Survived",kind="count")
plt.title("Survival Rate by Passenger Class")
plt.ylabel("Survival Rate")
plt.xlabel("Passenger Class")
plt.show()

In [ ]:
sns.histplot(data=train_df,x="Age",hue="Survived",kde=True)
plt.title("Distribution of Passenger Ages")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x="Age", y="Fare", hue="Survived", data=train_df, palette={0:"red", 1:"green"}, alpha=0.6)
plt.title("Fare vs Age (Colored by Survival)")
plt.xlabel("Age")
plt.ylabel("Fare")
plt.show()

I am uniting "SibSp" and "Parch" columns each other because they are connected each other and I can connect them under named of "Family_size"

In [ ]:
train_df["Family_size"] = train_df["SibSp"] + train_df["Parch"]
test_df["Family_size"] = test_df["SibSp"] + test_df["Parch"]

plt.figure(figsize=(8,4))
sns.barplot(x="Family_size", y="Survived", data=train_df, palette="mako")
plt.title("Survival Rate by Family Size")
plt.xlabel("Number of Family Members")
plt.ylabel("Survival Rate")
plt.show()

## 4- Preprocessing

In [ ]:
le=LabelEncoder()
train_df["Sex"] = le.fit_transform(train_df["Sex"])
test_df["Sex"] = le.fit_transform(test_df["Sex"])
train_df["Embarked"].fillna(train_df["Embarked"].mode()[0], inplace=True)
test_df["Embarked"].fillna(train_df["Embarked"].mode()[0], inplace=True)
train_df["Embarked"] = le.fit_transform(train_df["Embarked"])
test_df["Embarked"] = le.fit_transform(test_df["Embarked"])

In [ ]:
train_df.info()

In [ ]:
numeric_cols = train_df.select_dtypes(include=['int64', 'float64'])

corr = numeric_cols.corr()
plt.figure(figsize=(10, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap (Numeric Features)")
plt.show()

In [ ]:
train_df["Age"].fillna(train_df["Age"].median(), inplace=True)
train_df["Embarked"].fillna(train_df["Embarked"].mode()[0], inplace=True)
train_df["Fare"].fillna(train_df["Fare"].median(), inplace=True)
test_df["Age"].fillna(test_df["Age"].median(), inplace=True)
test_df["Embarked"].fillna(test_df["Embarked"].mode()[0], inplace=True)
test_df["Fare"].fillna(test_df["Fare"].median(), inplace=True)

## 5- Future Engineering

I am thinking Survived column could be related he is or she is alone in titanic so I created new column named "IsAlone" here.

In [ ]:
train_df["IsAlone"] = (train_df["Family_size"] == 1).astype(int)
test_df["IsAlone"] = (test_df["Family_size"] == 1).astype(int)

Here I realized that the while the cabin numbers of the economy passengers were not entered, the others were, so we cannot delete the cabin, but we can calculate a death rate by classifying it as having a cabin or not having a cabin.

In [ ]:
train_df["HasCabin"] = train_df["Cabin"].notnull().astype(int)
test_df["HasCabin"] = test_df["Cabin"].notnull().astype(int)

sns.barplot(x="HasCabin", y="Survived", data=train_df, palette="coolwarm")
plt.title("Survival Rate by Cabin Availability")
plt.show()

sns.barplot(x="Pclass", y="HasCabin", data=train_df)
plt.title("Cabin Info vs Passenger Class")
plt.show()

In [ ]:
train_df.drop(columns=["SibSp","Cabin", "Parch"], inplace=True)
train_df = train_df[train_df["Fare"] < 300]
test_df.loc[test_df["Fare"] > 300, "Fare"] =test_df["Fare"].median()
test_df.drop(columns=["SibSp","Cabin", "Parch"], inplace=True)

In [ ]:
train_df.head()

In [ ]:
x=train_df.drop(columns="Survived")
y=train_df["Survived"]

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,train_size=0.8,random_state=42)

## 6- LGBM Classification

In [ ]:
clf=lgb.LGBMClassifier(random_state=42,verbose=-1)
clf.fit(x_train, y_train)
clf.score(x_test,y_test)

## 7- with using GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'num_leaves': [15, 31, 63],
    'max_depth': [5, 10],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200]
}

grid = GridSearchCV(lgb.LGBMClassifier(random_state=42,verbose=-1), param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid.fit(x_train, y_train)
grid.score(x_test,y_test)

In [ ]:
train_df.info()

In [ ]:
test_df.info()

In [ ]:
x_train=train_df.drop(columns="Survived")
y_train=train_df["Survived"]
x_test = test_df.copy().drop(columns="PassengerId")

In [ ]:
y_test=grid.predict(x_test)
df_test = pd.DataFrame()
df_test["PassengerId"]=test_df["PassengerId"]
df_test["Survived"]=y_test

In [ ]:
df_test.info()

## Saving Our Data as CSV

In [ ]:
df_test.to_csv("results_with_lgb.csv", index=False) 

## Conclusion

The final model achieved an accuracy score of approximately 79.8%, which represents a solid baseline performance for the Titanic dataset. Key features contributing to survival prediction included Sex, Fare, and Age, which align with the historical narrative that women, children, and first-class passengers had higher survival chances.

--To further improve the results, future work could involve:

--creating new interaction features such as FamilySize or Title,

--using more advanced imputation methods (e.g., KNN Imputer),

--and experimenting with ensemble models to enhance prediction robustness.

Overall, this analysis demonstrates a complete end-to-end data science workflow — from exploratory data analysis to feature engineering, model training, and evaluation — showcasing how machine learning can be applied to gain insights from historical data.